# 🧠 RAG completo con Gemini

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 5 — 50 minutos**

---

## Objetivo

Construir un sistema RAG end-to-end: documentos → fragmentos → embeddings → ChromaDB → Gemini → respuesta fundamentada.

## Al terminar este bloque vas a poder:

1. Conectar ChromaDB como retriever con Gemini como generador.
2. Diseñar prompts RAG que fuercen al modelo a responder solo con el contexto.
3. Agregar documentos nuevos a un sistema RAG existente sin reiniciarlo.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **RAG completo** | Pipeline que une retrieval (ChromaDB) con generation (LLM). |
| **Retriever** | Componente que recibe una pregunta y devuelve los K chunks más relevantes. |
| **Context budget** | El límite de tokens que podemos inyectar en el prompt sin degradar al modelo. |
| **RetrievalQA** | Clase de LangChain que orquesta el pipeline RAG de principio a fin. |
| **MMR (Max Marginal Relevance)** | Estrategia de retrieval que evita chunks repetitivos entre sí. |

In [ ]:
# Instalamos las librerías necesarias para nuestro sistema RAG
# LangChain: la librería más popular para construir sistemas RAG de forma simple
# ChromaDB: nuestra base de datos vectorial para guardar los documentos
# Google GenerativeAI: para conectar con Gemini (solo para generación final)
# sentence-transformers: para embeddings locales multilenguaje
!pip install langchain langchain-google-genai langchain-chroma chromadb sentence-transformers -q

print("Todas las librerías instaladas correctamente")
print("IMPORTANTE: Solo se usará API de Gemini para generación final de respuestas")

In [ ]:
# Importamos todas las herramientas que vamos a necesitar
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from chromadb.utils import embedding_functions

# Configuración para mostrar mejor los resultados
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas exitosamente")
print("Usando embeddings locales para reducir uso de API")

## El sistema completo: ensamblando las piezas

### Analogía

Hasta acá tenías dos herramientas por separado: el buscador (ChromaDB) y el redactor (Gemini). RAG es el proceso que las conecta: el buscador encuentra los tres párrafos más relevantes, los pone frente al redactor, y el redactor escribe la respuesta mirando solo esos párrafos.

### Dónde vive esto en el mundo real

Este es el mismo patrón que usan los asistentes de documentos de Notion AI y los chatbots bancarios. El LLM solo puede responder con lo que está en tus documentos — eso hace las respuestas verificables y evita alucinaciones sobre tu dominio específico.

### Flujo del sistema

```
[Pregunta]  ->  ChromaDB.query(k=3)  ->  [3 chunks relevantes]
                                                 |
                         Gemini  <-  [prompt: contexto + pregunta]
                                                 |
                             [Respuesta fundamentada + fuentes]
```

### ✎ Para pensar

- ¿Por qué el prompt tiene la instrucción 'si no encontrás la información, decí que no sabés'?
- Si aumentás k de 3 a 10 chunks, ¿qué ventajas y qué problemas puede traer?

## Paso 1 — Documentos y fragmentación

El `RecursiveCharacterTextSplitter` divide por párrafo primero, luego por oración, luego por espacio — respetando coherencia semántica.

In [ ]:
# Creamos documentos de ejemplo sobre cultura y entretenimiento
# En la practica, estos datos vendrian de archivos PDF, Word, bases de datos, etc.

documentos_cultura = [
    {
        "titulo": "Guia de Cine Argentino Contemporaneo",
        "contenido": """
        Directores Destacados: Lucrecia Martel es considerada una de las directoras mas importantes de Latinoamerica.
        Sus peliculas como La Cienaga (2001) y Zama (2017) recibieron reconocimiento internacional.
        Damian Szifron gano el Goya a Mejor Pelicula Extranjera con Relatos Salvajes (2014).

        Festivales y Premios: El Festival de Mar del Plata es el unico festival de cine Clase A en Latinoamerica, reconocido por FIAPF.
        El BAFICI (Buenos Aires Festival Internacional de Cine Independiente) se realiza cada abril desde 1999.

        Nuevos Talentos: El cine argentino reciente destaca por peliculas como Argentina, 1985 (2022) que gano el Globo de Oro
        y fue nominada al Oscar. Directoras como Paula Hernandez y Celina Murga estan renovando el panorama cinematografico.
        """
    },
    {
        "titulo": "Rock Nacional: Historia y Figuras Clave",
        "contenido": """
        Origenes: El rock argentino comenzo en los 60 con bandas como Los Gatos y Almendra.
        Luis Alberto Spinetta es considerado el poeta del rock nacional. Murio en 2012 dejando un legado de mas de 40 años de carrera.

        Era Dorada: Los 80 vieron el auge con Soda Stereo, Patricio Rey y sus Redonditos de Ricota, Los Fabulosos Cadillacs.
        Gustavo Cerati es reconocido como uno de los musicos mas influyentes del rock en español.

        Actualidad: Bandas como Estelares, Las Pastillas del Abuelo, y la escena indie con El Mato un Policia Motorizado
        mantienen vigente el rock argentino. Los festivales como Cosquin Rock y Lollapalooza Argentina reunen multitudes cada año.
        """
    },
    {
        "titulo": "Videojuegos y Cultura Gamer en Argentina",
        "contenido": """
        Desarrollo Local: Argentina tiene una industria de videojuegos en crecimiento. Estudios como NGD Studios (Maestros del Mana)
        y Three Headed Monkey produjeron titulos que compitieron internacionalmente.

        Esports y Comunidad: Argentina cuenta con jugadores profesionales destacados en League of Legends, Counter-Strike y Dota 2.
        El Tecnopolis Gaming fue uno de los eventos mas grandes de gaming en Latinoamerica antes de la pandemia.

        Cultura Popular: Los streamers argentinos tienen millones de seguidores. La comunidad gamer se reune en eventos como
        Argentina Game Show (AGS), el evento gamer mas grande del pais, que se realiza anualmente en Buenos Aires.
        Juegos como Free Fire y FIFA son extremadamente populares entre jovenes argentinos.
        """
    },
    {
        "titulo": "Teatro y Cultura Porteña",
        "contenido": """
        Corriente Teatral: La Avenida Corrientes es conocida como la Broadway porteña. Concentra mas de 30 teatros
        y recibe 5 millones de espectadores anuales. El Teatro Colon es uno de los mejores teatros de opera del mundo
        por su acustica perfecta.

        Teatro Independiente: Buenos Aires tiene mas de 200 salas de teatro independiente. El barrio de Villa Crespo
        concentra muchas salas off, donde se experimenta con dramaturgia contemporanea.

        Festivales: El Festival Internacional de Buenos Aires (FIBA) se realiza bienalmente y trae compañias de todo el mundo.
        El Teatro San Martin ofrece funciones gratuitas y es referente de teatro nacional de calidad.
        """
    }
]

print(f"Se crearon {len(documentos_cultura)} documentos culturales de ejemplo:")
for i, doc in enumerate(documentos_cultura, 1):
    print(f"   {i}. {doc['titulo']}")

In [ ]:
# El "Text Splitter" es como un bibliotecario que divide documentos grandes
# en secciones manejables, manteniendo el contexto

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Cada fragmento tendrá máximo 500 caracteres
    chunk_overlap=50,      # 50 caracteres se superponen entre fragmentos para mantener contexto
    separators=["\n\n", "\n", ".", " "]  # Divide preferentemente por párrafos, luego oraciones
)

# Convertimos nuestros documentos al formato que entiende LangChain
documentos_langchain = []

for doc in documentos_cultura:
    # Cada documento se convierte en un objeto "Document" con contenido y metadata
    documento = Document(
        page_content=doc["contenido"],
        metadata={"titulo": doc["titulo"]}
    )
    documentos_langchain.append(documento)

# Dividimos todos los documentos en fragmentos más pequeños
fragmentos = text_splitter.split_documents(documentos_langchain)

print(f"✎ Documentos originales: {len(documentos_cultura)}")
print(f"🔪 Fragmentos creados: {len(fragmentos)}")
print(f"\n📋 Ejemplo de fragmento:")
print(f"Título: {fragmentos[0].metadata['titulo']}")
print(f"Contenido: {fragmentos[0].page_content[:200]}...")

In [ ]:
print(f"Título: {fragmentos[2].metadata['titulo']}")
print(f"Contenido: {fragmentos[2].page_content[:200]}...")

## Paso 2 — Embeddings locales y ChromaDB

`multilingual-e5-large` corre localmente (sin API). Gemini solo se usa para la generación final — ahorrás 70-80% de cuota.

In [ ]:
# Los "embeddings" convierten texto en vectores numéricos que representan el significado
# Usamos un modelo local multilenguaje que funciona excelente con español técnico
embeddings = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/multilingual-e5-large"  # Modelo multilenguaje optimizado para español
)

print("Modelo de embeddings local configurado (multilingual-e5-large)")
print("Ventaja: No consume cuota de API, solo procesamiento local")

In [ ]:
!pip install langchain_community -q

In [ ]:
# ChromaDB será nuestra "biblioteca inteligente" donde guardamos los vectores
# Es como un bibliotecario que puede encontrar libros por su tema, no solo por título
from langchain_community.embeddings import SentenceTransformerEmbeddings

embeddings = SentenceTransformerEmbeddings(model_name="intfloat/multilingual-e5-large")

vectorstore = Chroma.from_documents(
    documents=fragmentos,           # Los fragmentos de nuestros documentos
    embedding=embeddings,           # El modelo que convierte texto en vectores
    collection_name="documentos_empresa",  # Nombre de nuestra colección
    persist_directory="./chroma_db"  # Donde se guardan los datos (opcional)
)

print(f"Base de conocimiento vectorial creada con {len(fragmentos)} fragmentos")
print("El sistema ya puede buscar información por significado, no solo por palabras exactas")
print("Los embeddings se procesan localmente sin consumir cuota de Gemini")

## Paso 3 — Configuración de Gemini

`temperature=0.1` para respuestas precisas y consistentes.

In [ ]:
# Detectamos si estamos en Google Colab o en un entorno local
try:
    import google.colab
    from google.colab import userdata
    IN_COLAB = True
    print("❖ Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("❖ Entorno detectado: Local")

# Obtenemos la clave API según el entorno
if IN_COLAB:
    # En Colab: usar los secretos de Colab (más seguro)
    try:
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
        print("✓ Clave API cargada desde secretos de Colab")
    except Exception as e:
        print("✗ No se encontró GOOGLE_API_KEY en los secretos de Colab")
        print("   Ve a la barra lateral izquierda > 🔑 Secretos > Agregar GOOGLE_API_KEY")
        GOOGLE_API_KEY = input("Pega tu clave API de Google aquí: ")
else:
    # En local: usar variable de entorno
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
    if not GOOGLE_API_KEY:
        print("✗ No se encontró GOOGLE_API_KEY en las variables de entorno")
        print("   Opción 1: Agrega GOOGLE_API_KEY a tu archivo .env")
        print("   Opción 2: Ejecuta: export GOOGLE_API_KEY=tu_clave_aqui")
        GOOGLE_API_KEY = input("Pega tu clave API de Google aquí: ")
    else:
        print("✓ Clave API cargada desde variables de entorno")

# Configuramos la variable de entorno para que LangChain la use
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print("✦ Configuración de Gemini completada")

In [ ]:
# Configuramos el modelo Gemini que generará las respuestas finales
# NOTA: Solo este componente consume cuota de API, los embeddings son locales
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",    # Modelo rápido y eficiente de Gemini
    temperature=0.1,             # Baja creatividad = respuestas más precisas y consistentes
    google_api_key=GOOGLE_API_KEY
)

print("Modelo Gemini configurado")
print("   Modelo: gemini-1.5-flash (rápido y preciso)")
print("   Temperatura: 0.1 (respuestas consistentes y factuales)")
print("   IMPORTANTE: Solo la generación final usa API de Gemini")

## Paso 4 — Prompt template

El prompt le dice al modelo: usa SOLO el contexto provisto. Si no está ahí, decí que no sabés.

In [ ]:
# El prompt template es como las instrucciones que le damos a un asistente
# Le decimos exactamente como debe comportarse y que formato usar

template_respuesta = """
Sos un asistente experto en cultura, entretenimiento y artes.
Tu trabajo es responder preguntas basandote UNICAMENTE en la informacion
proporcionada en los documentos culturales.

INSTRUCCIONES IMPORTANTES:
1. Solo usa informacion que aparece explicitamente en los documentos
2. Si no encontras la informacion especifica, decilo claramente
3. Cita el documento o seccion cuando sea posible
4. Se preciso con nombres, fechas y datos
5. Usa un tono amigable pero informado

CONTEXTO DE LOS DOCUMENTOS:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:
"""

# Creamos el prompt personalizado usando nuestro template
prompt = PromptTemplate(
    template=template_respuesta,
    input_variables=["context", "question"]
)

print("Template de respuesta configurado")
print("El asistente seguira instrucciones especificas para dar respuestas precisas")

## Paso 5 — Ensamblado del pipeline

`RetrievalQA` conecta ChromaDB + Gemini + prompt template. Con `return_source_documents=True` podés mostrar qué fragmentos se usaron.

In [ ]:
# El "RetrievalQA" es el corazón de nuestro sistema RAG
# Conecta la búsqueda (Retrieval) con la generación (QA = Question Answering)

sistema_rag = RetrievalQA.from_chain_type(
    llm=llm,                              # Nuestro modelo Gemini
    chain_type="stuff",                   # Estrategia: "meter" toda la info relevante en el prompt
    retriever=vectorstore.as_retriever(   # Configuración del buscador
        search_kwargs={"k": 3}            # Buscar los 3 fragmentos más relevantes
    ),
    chain_type_kwargs={                   # Configuraciones adicionales
        "prompt": prompt,                 # Nuestras instrucciones personalizadas
        "verbose": False                  # No mostrar pasos internos (para mantenerlo limpio)
    },
    return_source_documents=True          # Devolver también los documentos fuente
)

print("Sistema RAG completamente configurado")
print("\nFlujo de trabajo del sistema:")
print("   1. Usuario hace una pregunta")
print("   2. El sistema busca los 3 fragmentos más relevantes (LOCAL)")
print("   3. Gemini lee esos fragmentos y genera una respuesta (API)")
print("   4. Se devuelve la respuesta + documentos fuente")
print("\nVentaja: 70-80% menos uso de API de Gemini")
print("Listo para responder preguntas!")

## Probando el sistema

Las primeras cuatro preguntas están en los documentos. La quinta está fuera del scope — observá cómo responde el sistema.

In [ ]:
# Función auxiliar para mostrar respuestas de forma clara y educativa
def hacer_pregunta(pregunta, mostrar_fuentes=True):
    """
    Función que procesa una pregunta y muestra la respuesta de forma educativa

    Args:
        pregunta (str): La pregunta que queremos hacer al sistema
        mostrar_fuentes (bool): Si mostrar o no los documentos fuente
    """
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {pregunta}")
    print(f"{'='*60}")

    # Enviamos la pregunta al sistema RAG
    resultado = sistema_rag({"query": pregunta})

    # Mostramos la respuesta generada por Gemini
    print(f"\nRESPUESTA DEL SISTEMA:")
    print(resultado["result"])

    # Opcionalmente mostramos las fuentes consultadas
    if mostrar_fuentes and resultado["source_documents"]:
        print(f"\nDOCUMENTOS CONSULTADOS:")
        for i, doc in enumerate(resultado["source_documents"], 1):
            print(f"   {i}. {doc.metadata['titulo']}")
            print(f"      Fragmento: {doc.page_content[:100]}...")

    return resultado

print("Función de prueba lista")
print("Ahora podemos hacer preguntas específicas sobre nuestros documentos culturales")

In [ ]:
# Preguntamos sobre la política de vacaciones
resultado1 = hacer_pregunta("¿Quien es Lucrecia Martel?")

In [ ]:
# Preguntamos sobre requerimientos de contraseñas
resultado2 = hacer_pregunta("¿Cuales fueron los hits de Soda Stereo?")

In [ ]:
# Preguntamos sobre gastos de capacitación
resultado3 = hacer_pregunta("¿Se producen video juegos en Argentina?")

In [ ]:
# Preguntamos sobre contactos internos
resultado4 = hacer_pregunta("¿Cuantas salas de teatro hay en Buenos Aires?")

In [ ]:
# Probamos qué pasa cuando preguntamos algo que no está en nuestros documentos
resultado5 = hacer_pregunta("¿Cuál es el procedimiento para solicitar una computadora nueva?")

### ✎ Para pensar

- ¿Cómo manejó el sistema la pregunta fuera del scope? ¿Inventó algo o admitió que no sabía?
- Si quisieras que el sistema cite explicitamente el documento fuente en cada respuesta, ¿qué cambiarías en el prompt template?

## Actualización dinámica

Una ventaja clave de RAG: podés agregar documentos nuevos sin reiniciar ni reentrenar. El sistema los indexa y los usa en la próxima consulta.

In [ ]:
# Simulamos que llegan nuevos documentos culturales
nuevos_documentos = [
    {
        "titulo": "Musica Urbana Argentina: Trap y Hip Hop",
        "contenido": """
        Referentes: Duki, Bizarrap y Nicki Nicole son algunos de los mayores exponentes del trap argentino.
        Duki lleno el estadio de Velez dos veces consecutivas en 2022, marcando un hito historico.
        Bizarrap llego a 50 millones de suscriptores en YouTube con sus Music Sessions.

        Escena Local: El trap argentino nacio en plazas y cypher callejeros. La escena se profesionalizo
        en la ultima decada con productoras y sellos discograficos dedicados al genero.

        Impacto Internacional: Artistas argentinos colaboran con figuras internacionales.
        Bizarrap produjo sesiones con Shakira, Residente y Quevedo que rompieron records de reproduccion.
        """
    },
    {
        "titulo": "Literatura Argentina Contemporanea",
        "contenido": """
        Autores Destacados: Claudia Piñeiro, Samanta Schweblin y Mariana Enriquez son referentes de la literatura
        argentina actual. Sus obras se traducen a multiples idiomas y ganan premios internacionales.

        Feria del Libro: La Feria del Libro de Buenos Aires es una de las mas importantes de habla hispana.
        Se realiza anualmente en La Rural y recibe mas de un millon de visitantes.

        Premios: Argentina cuenta con premios literarios importantes como el Clarin de Novela y el Premio Konex.
        Escritores argentinos fueron galardonados con el Premio Cervantes, maximo reconocimiento en español.
        """
    }
]

print(f"Llegaron {len(nuevos_documentos)} documentos nuevos:")
for doc in nuevos_documentos:
    print(f"   - {doc['titulo']}")

In [ ]:
# Función para agregar nuevos documentos al sistema existente
def agregar_documentos(nuevos_docs, vectorstore):
    """
    Agrega nuevos documentos a nuestro sistema RAG existente
    """
    print("🔄 Procesando nuevos documentos...")

    # Convertir a formato LangChain
    docs_langchain = []
    for doc in nuevos_docs:
        documento = Document(
            page_content=doc["contenido"],
            metadata={"titulo": doc["titulo"]}
        )
        docs_langchain.append(documento)

    # Dividir en fragmentos
    nuevos_fragmentos = text_splitter.split_documents(docs_langchain)
    print(f"✎ Se crearon {len(nuevos_fragmentos)} nuevos fragmentos")

    # Agregar al vectorstore existente
    vectorstore.add_documents(nuevos_fragmentos)
    print(f"✓ Documentos agregados exitosamente a la base de conocimiento")

    return len(nuevos_fragmentos)

# Agregamos los nuevos documentos
fragmentos_nuevos = agregar_documentos(nuevos_documentos, vectorstore)

print(f"\n📊 Estado actual del sistema:")
print(f"   Documentos originales: {len(documentos_cultura)} docs")
print(f"   Documentos nuevos: {len(nuevos_documentos)} docs")
print(f"   Total de fragmentos en la base: {len(fragmentos) + fragmentos_nuevos} fragmentos")

In [ ]:
# Probemos preguntas sobre los nuevos documentos
print("🆕 Probando preguntas sobre los documentos recién agregados:\n")

hacer_pregunta("¿Qué días son obligatorios ir a la oficina?")
hacer_pregunta("¿Qué libros escribio Samanta Schweblin?")
hacer_pregunta("¿Quien es Bizarrap?")

In [ ]:
# --- Espacio de practica ---
#
# Construi tu propio mini-RAG:
#   1. Crea 3-4 documentos sobre un tema que te interese
#   2. Agrega al vectorstore con vectorstore.add_documents()
#   3. Hace 3 preguntas: 2 que esten en scope y 1 que no
#   4. Observa como el sistema usa las fuentes
#
# Bonus: modifica el template para que cite siempre el titulo del documento
#
from langchain.schema import Document

mis_docs = [
    Document(page_content="Tu contenido 1", metadata={"titulo": "Mi doc 1"}),
    Document(page_content="Tu contenido 2", metadata={"titulo": "Mi doc 2"}),
]

fragmentos_nuevos = text_splitter.split_documents(mis_docs)
vectorstore.add_documents(fragmentos_nuevos)

resultado = hacer_pregunta("Tu pregunta sobre el tema")

## Cierre del bloque y de la cursada

| Componente RAG | Qué hace |
|---|---|
| **Document Loaders** | Extraen texto de PDFs, webs, etc. |
| **Text Splitter** | Fragmenta en chunks coherentes con overlap |
| **Embeddings locales** | Vectorizan los chunks sin consumir API |
| **ChromaDB** | Almacena y busca vectores por similitud semántica |
| **Prompt template** | Define el contrato entre el sistema y el LLM |
| **RetrievalQA** | Orquesta todo el pipeline |
| **Gemini** | Genera la respuesta final fundamentada en el contexto |

### Lo que construiste en estos dos encuentros

Arrancaste entendiendo cómo un LLM lee texto (tokens → embeddings) y terminaste construyendo un sistema que responde preguntas sobre tus propios documentos. Ese camino — del concepto al sistema funcional — es exactamente lo que hacen los equipos de IA en producción.